# PYZX Tutorial

## Generating Circuits

In [1]:
import pyzx
from pyzx.generate import cliffords, cnots

# for phase free circuits
# circ = cnots(qubits=10, depth=20)

# for clifford circuits
circ = cliffords(qubits=10, depth=20)

# pyzx works in-place, so keep a copy of the original graph
circ_copy = circ.copy()

## Drawing

In [2]:
from pyzx import draw

draw(circ)
print(circ.stats())

Graph(80 vertices, 76 edges)
degree distribution: 
1: 20
2: 48
3: 12



## Basic Clean up (for input graph)

In [3]:
# to graph form
from pyzx import to_gh
to_gh(circ)
draw(circ)

In [4]:
# fuse and remove spiders
from pyzx import spider_simp, id_simp

spider_simp(circ)
id_simp(circ)
draw(circ)

spider_simp: 13. 7. 4. 1. 1. 1.  6 iterations
id_simp: 12. 2.  2 iterations


## Simplify graph using pyzx

In [5]:
from pyzx import full_reduce, clifford_simp, phase_free_simp

full_reduce(circ)
draw(circ)
print(circ.stats())

Graph(35 vertices, 36 edges)
degree distribution: 
1: 20
2: 7
3: 2
4: 1
5: 3
6: 1
7: 1



## Verification

In [6]:
# (U.adjoint) o U -> Identity
circ_combine = circ.adjoint() + circ_copy.copy()

full_reduce(circ_combine)
draw(circ_combine)

## Some useful methods

In [7]:
circ = cliffords(qubits=10, depth=20)

draw(circ)
to_gh(circ)
draw(circ)
vs = list(circ.vertex_set())
es = list(circ.edge_set())

In [8]:
# Vertex attributes

# 0: VertexType.BOUNDARY
# 1: VertexType.Z
# 2: VertexType.X (these won't come up if you do to_gh first to input)
# 3: VertexType.H_BOX (these won't come up)

from pyzx import VertexType
print(VertexType.Type)
circ.types()[vs[0]]

typing.Literal[0, 1, 2, 3]


0

In [9]:
# Edge attributes

# The edges to the boundary can be normal or Hadamard edges
# after spider_simp, edges between Z vertices are always Hadamard edges

# 1: EdgeType.SIMPLE
# 2: EdgeType.HADAMARD

from pyzx import EdgeType
print(EdgeType.Type)
circ.edge_type(es[0])

typing.Literal[1, 2]


1

In [10]:
# Phases are stored as multiples of Pi

circ.phases()

{0: 0,
 1: 0,
 2: 0,
 3: 0,
 4: 0,
 5: 0,
 6: 0,
 7: 0,
 8: 0,
 9: 0,
 10: 0,
 11: 0,
 12: 0,
 13: 0,
 14: 0,
 15: 0,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: Fraction(1, 2),
 21: Fraction(0, 1),
 22: 0,
 23: 0,
 24: Fraction(0, 1),
 25: Fraction(1, 2),
 26: Fraction(1, 2),
 27: Fraction(3, 2),
 28: 0,
 29: Fraction(0, 1),
 30: 0,
 31: 0,
 32: Fraction(3, 2),
 33: Fraction(1, 1),
 34: Fraction(3, 2),
 35: Fraction(1, 1),
 36: Fraction(3, 2),
 37: Fraction(3, 2),
 38: Fraction(3, 2),
 39: Fraction(3, 2),
 40: Fraction(0, 1),
 41: Fraction(0, 1),
 42: Fraction(3, 2),
 43: Fraction(0, 1),
 44: 0,
 45: Fraction(3, 2),
 46: Fraction(0, 1),
 47: Fraction(1, 2),
 48: Fraction(3, 2),
 49: Fraction(1, 1),
 50: Fraction(1, 1),
 51: Fraction(1, 2),
 52: Fraction(3, 2),
 53: Fraction(0, 1),
 54: Fraction(1, 1),
 55: Fraction(1, 1),
 56: 0,
 57: Fraction(0, 1),
 58: Fraction(3, 2),
 59: Fraction(0, 1),
 60: 0,
 61: 0,
 62: 0,
 63: 0,
 64: 0,
 65: 0,
 66: 0,
 67: 0,
 68: 0,
 69: 0,
 70: 0,
 71: 0,
 72: 

In [11]:
from collections import OrderedDict
# old encoding: node types | phases | edge list
# new encoding: node types + phases | edge list

def to_tokens(graph):
    """ Converts pyzx graph into tokens. """
    indices = sorted(graph.types().keys())
    types_dict = OrderedDict(graph.types())
    phases_dict = OrderedDict(graph.phases())
    
    # store phases as multiples of 4
    node_type_and_phases = [
        f"NT{typ}|4P{int(phase * 4)}"
        for typ, phase in zip(types_dict.values(), phases_dict.values())]
    # edges: (type, n_i, n_j)
    edge_types = [f"ET{graph.edge_type(edge)}" for edge in graph.edges()]
    edges = [f"EP{indices.index(x)}" for edge in graph.edges() for x in edge]

    tokens = node_type_and_phases + edge_types + edges
    return tokens

to_tokens(circ)

['NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P2',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P2',
 'NT1|4P2',
 'NT1|4P6',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P6',
 'NT1|4P4',
 'NT1|4P6',
 'NT1|4P4',
 'NT1|4P6',
 'NT1|4P6',
 'NT1|4P6',
 'NT1|4P6',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P6',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P6',
 'NT1|4P0',
 'NT1|4P2',
 'NT1|4P6',
 'NT1|4P4',
 'NT1|4P4',
 'NT1|4P2',
 'NT1|4P6',
 'NT1|4P0',
 'NT1|4P4',
 'NT1|4P4',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P6',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT1|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'NT0|4P0',
 'ET1',
 'ET1',
 'ET1',
 'ET1',
 'ET1',


In [12]:
from pyzx import Graph
from random import random

def from_tokens(tokens):
    """ Deserialise tokens into pyzx graph, with random positioning. """
    graph = Graph()
    
    node_types = [int(token.split("|")[0][2:]) for token in tokens if token.startswith("NT")]
    phases = [int(token.split("|")[1][2:]) for token in tokens if token.startswith("NT")]
    edge_types = [int(token[2:]) for token in tokens if token.startswith("ET")]
    edges = [int(token[2:]) for token in tokens if token.startswith("EP")]

    seen_z = False
    inputs = []
    outputs = []

    assert all([typ in [0, 1] for typ in node_types])
    for typ, phase in zip(node_types, phases):
        v = graph.add_vertex(phase=phase / 4, ty=typ)
        if typ == 0:
            if not seen_z:
                graph.set_position(v, q=len(inputs), r=0)
                inputs.append(v)
            else:
                graph.set_position(v, q=len(outputs), r=len(inputs))
                outputs.append(v)
        else:
            seen_z = True
            x = 0.5 + random() * (len(inputs) - 1)
            y = 0.5 + random() * (len(inputs) - 1)
            graph.set_position(v, q=x, r=y)

    graph.set_inputs(inputs)
    graph.set_outputs(outputs)
    for typ, edge in zip(edge_types, zip(edges[::2], edges[1::2])):
        graph.add_edge(edge, typ)

    return graph

In [13]:
z = circ + from_tokens(to_tokens(circ)).adjoint()
full_reduce(z)
draw(z)

In [14]:
draw(from_tokens(to_tokens(circ)))

In [15]:
'''
Calling the generator and decoder from the notebook
'''

from src.envs.zx import *
import numpy as np
import pyzx

# mandatory parameters for the environment
class Params:
    def __init__(self):
        self.max_len=500
        self.operation = "simplify"
        self.min_qubits = 3
        self.max_qubits = 11
        self.min_depth=10
        self.max_depth = 20
        self.max_int=200
        self.max_nodes = 100

params = Params()

# initialize the env, and its rng (one different rng per instance, because of multipocessing)
env = ZXEnvironment(params)
env.rng=np.random

# generate an instance
x,y = env.gen_expr()
# input graph
input_graph = env.encoder.decode(x)
print(x)
print()
print(input_graph)
print (input_graph.stats())
pyzx.draw(input_graph)

# output graph
output_graph= env.encoder.decode(y)
print (y)
print(output_graph.stats())
pyzx.draw(output_graph)

assert x == env.encoder.encode(input_graph)
assert y == env.encoder.encode(output_graph)
#print(env.encoder.encode(output_graph))

['52', '53', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '6', '1', '6', '1', '0', '1', '0', '1', '2', '1', '4', '1', '2', '1', '0', '1', '4', '1', '0', '1', '2', '1', '0', '1', '4', '1', '2', '1', '4', '1', '0', '1', '4', '1', '6', '1', '6', '1', '4', '1', '0', '1', '0', '1', '2', '1', '6', '1', '4', '1', '0', '1', '4', '1', '6', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', 'N0', 'N5', '1', 'N1', 'N6', '1', 'N2', 'N7', '1', 'N3', 'N8', '1', 'N4', 'N9', '1', 'N5', 'N12', '1', 'N6', 'N14', '2', 'N7', 'N19', '2', 'N8', 'N11', '1', 'N9', 'N10', '2', 'N10', 'N13', '1', 'N11', 'N15', '2', 'N12', 'N13', '1', 'N12', 'N16', '1', 'N13', 'N17', '2', 'N14', 'N18', '1', 'N15', 'N25', '2', 'N16', 'N20', '2', 'N17', 'N21', '2', 'N18', 'N19', '1', 'N18', 'N22', '2', 'N19', 'N26', '2', 'N20', 'N21', '2', 'N20', 'N23', '1', 'N21', 'N29', '1', 'N

['18', '19', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '2', '1', '0', '1', '0', '1', '2', '1', '4', '1', '4', '1', '4', '1', '2', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '2', 'N0', 'N7', '1', 'N1', 'N5', '1', 'N2', 'N15', '1', 'N3', 'N11', '1', 'N4', 'N6', '2', 'N5', 'N8', '2', 'N6', 'N7', '2', 'N6', 'N11', '2', 'N6', 'N12', '2', 'N6', 'N9', '2', 'N7', 'N9', '1', 'N8', 'N14', '2', 'N8', 'N10', '2', 'N9', 'N17', '2', 'N10', 'N11', '2', 'N10', 'N13', '2', 'N10', 'N12', '2', 'N11', 'N12', '2', 'N12', 'N16']
Graph(18 vertices, 19 edges)
degree distribution: 
1: 10
2: 1
3: 3
4: 3
5: 1



In [16]:
''' statistics on graph lengths'''

from src.envs.zx import *
import numpy as np
import pyzx

# mandatory parameters for the environment
class Params:
    def __init__(self):
        self.max_len=5000
        self.operation = "simplify"
        self.min_qubits = 15
        self.max_qubits = 15
        self.min_depth=30
        self.max_depth = 30
        self.max_int=1000
        self.max_nodes = 1000

params = Params()

# initialize the env, and its rng (one different rng per instance, because of multipocessing)
env = ZXEnvironment(params)
env.rng=np.random

# generate an instance
x_len = []
y_len = []
# print(env.encoder.symbols)
for _ in range(3000):
    x,y = env.gen_expr()
    #for t in x:
    #    if not t in env.encoder.symbols:
    #        print(t,"unfound")
    #for t in y:
    #    if not t in env.encoder.symbols:
    #        print(t,"not found")
    x_len.append(len(x))
    y_len.append(len(y))
    
print(f"x: avg {np.mean(x_len)}, std_dev: {np.std(x_len)}, 3 stddev: {np.mean(x_len)+3*np.std(x_len)}")
print(f"y: avg {np.mean(y_len)}, std_dev: {np.std(y_len)}, 3 stddev: {np.mean(y_len)+3*np.std(y_len)}")


x: avg 592.891, std_dev: 7.899564481666062, 3 stddev: 616.5896934449981
y: avg 280.9843333333333, std_dev: 26.448971395668472, 3 stddev: 360.3312475203387
